# Lab 05 — Object Versioning

S3 **versioning** keeps every write to an object as a separate, addressable version.
Without versioning, uploading a new file with the same key silently overwrites the old one.
With versioning enabled, every `put_object` creates a **new version** alongside the previous ones.

### What you'll practice

| # | Operation | API |
|---|-----------|-----|
| 1 | Enable versioning on a bucket | `put_bucket_versioning` |
| 2 | Write three versions of the same key | `put_object` |
| 3 | List all versions | `list_object_versions` |
| 4 | Soft-delete (create a delete marker) | `delete_object` |
| 5 | Access a specific historical version | `get_object(VersionId=...)` |
| 6 | Hard-delete all versions | `delete_object` per version |

### Delete markers — what are they?

When you call `delete_object` on a versioned bucket **without** specifying a `VersionId`,
S3 does **not** delete the object — it inserts a **delete marker** as the new "current" version.
The object appears deleted to regular `get_object` calls, but all previous versions are still
there. To truly delete an object, you must delete each version and marker individually.


---
## 0 · Prerequisites

1. RustFS is running: `docker compose up -d`
2. Virtual environment active with `boto3` installed: `uv sync`


---
## Setup — Connect and Create a Versioned Bucket


In [ ]:
import boto3
from botocore.config import Config

s3 = boto3.client(
    service_name='s3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='adminpassword',
    region_name='us-east-1',
    use_ssl=False,
    config=Config(
        signature_version='s3v4',
        s3={'addressing_style': 'path'},
    ),
)

BUCKET = 'lab5-versions'
KEY = 'docs/report.txt'

# Create the bucket if needed
existing = {b['Name'] for b in s3.list_buckets()['Buckets']}
if BUCKET not in existing:
    s3.create_bucket(Bucket=BUCKET)
    print(f'✅ Created bucket: {BUCKET}')
else:
    print(f'✅ Bucket already exists: {BUCKET}')


---
## 1 · Enable Versioning

Versioning must be explicitly enabled. Once enabled, it can be **suspended** but never
fully disabled — existing versions remain accessible forever.


In [ ]:
s3.put_bucket_versioning(
    Bucket=BUCKET,
    VersioningConfiguration={'Status': 'Enabled'},
)

# Confirm status
status = s3.get_bucket_versioning(Bucket=BUCKET)
print(f'✅ Versioning status: {status.get("Status", "Disabled")}')


---
## 2 · Write Version 1

The first `put_object` creates an object with a unique `VersionId`.


In [ ]:
resp = s3.put_object(
    Bucket=BUCKET, Key=KEY,
    Body=b'Report v1: Q1 2025: initial draft.',
)
v1_id = resp['VersionId']

# Read it back to confirm
content = s3.get_object(Bucket=BUCKET, Key=KEY)['Body'].read().decode()
print('Version 1 written:')
print(f'  VersionId: {v1_id}')
print(f'  Content:   {content}')


---
## 3 · Overwrite — Version 2

Uploading to the same key creates a new version. The previous version is preserved.


In [ ]:
resp = s3.put_object(
    Bucket=BUCKET, Key=KEY,
    Body=b'Report v2: Q1 2025: peer review applied.',
)
v2_id = resp['VersionId']

content = s3.get_object(Bucket=BUCKET, Key=KEY)['Body'].read().decode()
print('Version 2 written:')
print(f'  VersionId: {v2_id}')
print(f'  Content:   {content}')


---
## 4 · Overwrite — Version 3 (final)


In [ ]:
resp = s3.put_object(
    Bucket=BUCKET, Key=KEY,
    Body=b'Report v3: Q1 2025: approved and published.',
)
v3_id = resp['VersionId']

content = s3.get_object(Bucket=BUCKET, Key=KEY)['Body'].read().decode()
print('Version 3 written (latest):')
print(f'  VersionId: {v3_id}')
print(f'  Content:   {content}')


---
## 5 · List All Versions

`list_object_versions` returns every version and delete marker for every key in a bucket.
The version with `IsLatest: True` is what a plain `get_object` would return.


In [ ]:
response = s3.list_object_versions(Bucket=BUCKET)
versions = response.get('Versions', [])

print(f'All versions of s3://{BUCKET}/{KEY}:')
print(f'  {"VersionId":>32}  {"IsLatest":>8}  {"Modified":>26}  Content')
print('  ' + '-' * 90)

for v in versions:
    # Fetch the content to display alongside metadata
    content = s3.get_object(
        Bucket=BUCKET, Key=v['Key'], VersionId=v['VersionId']
    )['Body'].read().decode()
    latest = '*** latest' if v['IsLatest'] else ''
    print(f"  {v['VersionId']:>32}  {str(v['IsLatest']):>8}  {str(v['LastModified']):>26}  {content}  {latest}")


---
## 6 · Soft Delete — Insert a Delete Marker

Calling `delete_object` **without** a `VersionId` does not remove any data.
It inserts a **delete marker** as the new current version, making the object
appear deleted to unversioned clients.


In [ ]:
resp = s3.delete_object(Bucket=BUCKET, Key=KEY)
marker_id = resp.get('VersionId')

print('Delete marker created:')
print(f'  VersionId:   {marker_id}')
print(f'  DeleteMarker: {resp.get("DeleteMarker")}')
print()

# Verify: unversioned read now fails
try:
    s3.get_object(Bucket=BUCKET, Key=KEY)
    print('Object still accessible (unexpected)')
except s3.exceptions.NoSuchKey:
    print('Unversioned read → NoSuchKey (expected: delete marker hides the object)')
except Exception as e:
    print(f'Read failed as expected: {type(e).__name__}: {e}')


---
## 7 · Restore a Previous Version

Accessing a specific version is as simple as passing `VersionId` to `get_object`.
To "restore" a version as the new current, upload its content as a new `put_object`.


In [ ]:
# List versions again — now includes the delete marker
response = s3.list_object_versions(Bucket=BUCKET)

print('Current state (marker + versions):')
for marker in response.get('DeleteMarkers', []):
    print(f"  [DELETE MARKER] VersionId={marker['VersionId']}  IsLatest={marker['IsLatest']}")
for v in response.get('Versions', []):
    print(f"  [VERSION]       VersionId={v['VersionId']}  IsLatest={v['IsLatest']}")

print()

# Access version 1 directly by its VersionId
v1_content = s3.get_object(
    Bucket=BUCKET, Key=KEY, VersionId=v1_id
)['Body'].read().decode()
print(f'Version 1 content (read by VersionId): {v1_content}')

# To restore v1 as the new current: just re-upload its content
s3.put_object(Bucket=BUCKET, Key=KEY, Body=v1_content.encode())
current = s3.get_object(Bucket=BUCKET, Key=KEY)['Body'].read().decode()
print(f'Restored to current:                   {current}')


---
## 8 · Hard Delete — Purge All Versions

To permanently delete all data for a key, you must explicitly delete every version
and every delete marker. There is no "delete all versions" API — you iterate and
delete each one by `VersionId`.


In [ ]:
response = s3.list_object_versions(Bucket=BUCKET)

# Delete all object versions
for v in response.get('Versions', []):
    s3.delete_object(Bucket=BUCKET, Key=v['Key'], VersionId=v['VersionId'])
    print(f'  Deleted version  {v["VersionId"]}')

# Delete all delete markers
for m in response.get('DeleteMarkers', []):
    s3.delete_object(Bucket=BUCKET, Key=m['Key'], VersionId=m['VersionId'])
    print(f'  Deleted marker   {m["VersionId"]}')

# Verify bucket is empty
remaining = s3.list_object_versions(Bucket=BUCKET)
total_remaining = len(remaining.get('Versions', [])) + len(remaining.get('DeleteMarkers', []))
print(f'\nRemaining versions/markers: {total_remaining}')
print('✅ All versions permanently deleted.' if total_remaining == 0 else '❌ Some versions remain!')


---
## 9 · Cleanup


In [ ]:
s3.delete_bucket(Bucket=BUCKET)
print(f'🗑️  Deleted bucket: {BUCKET}')
print('\n✅ Cleanup complete!')


---
## 📋 Summary

| Concept | Behavior |
|---------|----------|
| `put_object` (versioning on) | Creates a new version; previous version preserved |
| `get_object` (no VersionId) | Returns the **latest** version |
| `get_object(VersionId=...)` | Returns that specific historical version |
| `delete_object` (no VersionId) | Inserts a delete marker; data NOT deleted |
| `delete_object(VersionId=...)` | Permanently deletes that specific version |
| `list_object_versions` | Returns all versions AND delete markers |

### When to use versioning

- **Audit trail**: regulations require keeping every version of a file
- **Accidental overwrite protection**: recover the previous version of a Gold table
  that was overwritten by a buggy pipeline
- **Blue/green data deploys**: publish a new Gold table version while keeping the old
  one accessible until consumers have migrated

### Next steps

- **Lab 06**: Erasure Coding deep dive — storage efficiency vs fault tolerance
